# 🌲 02 — Drishti: Train GBT Delay Predictor (Spark MLlib + MLflow)

This is the **ML core of Drishti**.

**Why GBTRegressor:**
- Native Spark MLlib → distributed training, checks track's "Spark MLlib" technical hook
- Handles mixed categorical + numeric features via StringIndexer + VectorAssembler
- Beats linear models by ~30% on tabular data with non-linear interactions (e.g., fog × peak-hour)
- SHAP works natively on tree-based models → interpretable explanations for the demo
- CPU-native, no GPU needed

**MLflow integration:** every run is logged with params, metrics, the model artifact, and registered to Unity Catalog as `setu_rail.gold.setu_delay_predictor`.

**Target metrics:** MAE ≤ 12 min, RMSE ≤ 22 min.

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, when, isnan, isnull, coalesce

mlflow.set_registry_uri("databricks-uc")   # Unity Catalog model registry
EXPERIMENT = "/Shared/setu-rail-drishti"
mlflow.set_experiment(EXPERIMENT)
print(f"MLflow experiment: {EXPERIMENT}")

## Step 0 — Inspect the feature table to understand column names

In [0]:
# ── CELL 0: Nuclear cache clear — run this FIRST, alone ──────
import gc

# Delete every possible model variable
for var in ['model', 'pipeline', 'preds', 'train_df', 'test_df', 
            'df_filled', 'df_enc', 'df_clean', 'assembler', 'gbt']:
    try:
        exec(f"del {var}")
        print(f"🗑️  Deleted {var}")
    except:
        pass

gc.collect()
print("✅ GC done")

# Drop temp tables if they exist
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._train_tmp")
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._test_tmp")
print("✅ Temp tables dropped")

In [0]:
# Load and inspect the ML-ready feature table
df = spark.table("setu_rail.gold.features_delay_ml")

print(f"Total rows available: {df.count():,}")
print(f"\nColumns in the table:")
df.printSchema()

print(f"\nFirst few rows:")
df.show(3, truncate=False)

## Step 1 — Data preparation with intelligent null handling

In [0]:
# List of NUMERIC features that should exist in the table
# These are the real features from 02_build_silver_enriched.ipynb
NUMERIC_FEATURES = [
    "stop_seq",                   # 0-indexed stop sequence
    "total_stops",                # count of stops
    "cumulative_travel_min",      # minutes from origin departure
    "dwell_min",                  # arrival to departure time at station
    "scheduled_hour",             # hour of departure (0-23)
    "is_peak_hour",               # 1 if peak, 0 otherwise
    "pm25",                       # air quality proxy
    "no2",                        # another air quality metric
    "journey_day",                # day of multi-day journey
]

CATEGORICAL_FEATURES = [
    "train_no",
    "station_code",
    "train_type",
    "zone",
]

LABEL = "arrival_delay_min"

# Check which columns exist
existing_cols = set(df.columns)
print("Checking feature availability:")
print(f"Numeric features available: {[f for f in NUMERIC_FEATURES if f in existing_cols]}")
print(f"Categorical features available: {[f for f in CATEGORICAL_FEATURES if f in existing_cols]}")
print(f"Label available: {LABEL in existing_cols}")

# Filter to only features that exist
numeric_features_avail = [f for f in NUMERIC_FEATURES if f in existing_cols]
categorical_features_avail = [f for f in CATEGORICAL_FEATURES if f in existing_cols]

print(f"\nWill use {len(numeric_features_avail)} numeric + {len(categorical_features_avail)} categorical features")

In [0]:
# Prepare data: drop rows with null label, fill nulls in features with sensible defaults
df_clean = df.filter(col(LABEL).isNotNull())

# Fill numeric features with column median or defaults
fill_dict = {
    "stop_seq": 0,
    "total_stops": 50,
    "cumulative_travel_min": 0,
    "dwell_min": 5,
    "scheduled_hour": 12,
    "is_peak_hour": 0,
    "pm25": 60.0,
    "no2": 25.0,
    "journey_day": 1,
    "latitude": 20.0,
    "longitude": 77.0,
}

# Only fill columns that exist
actual_fill = {k: v for k, v in fill_dict.items() if k in df_clean.columns}
df_clean = df_clean.fillna(actual_fill)

# For categorical, fill with 'UNKNOWN'
cat_cols_fill = {f: "UNKNOWN" for f in categorical_features_avail if f in df_clean.columns}
df_clean = df_clean.fillna(cat_cols_fill)

print(f"Rows after cleaning: {df_clean.count():,}")
df_clean.show(2, truncate=False)

## Step 2 — Time-respecting train/test split

In [0]:
# If we have a month column, split by month (temporal integrity)
if "month" in df_clean.columns:
    train_df = df_clean.filter(col("month") <= 9)
    test_df  = df_clean.filter(col("month") > 9)
    print(f"Split by month: train (1-9) vs test (10-12)")
else:
    # Otherwise use random 80/20
    train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
    print(f"Random 80/20 split (no time column)")

print(f"Train rows: {train_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

## Step 3 — Build and train the ML Pipeline

In [0]:
# ═══════════════════════════════════════════════════════════════
# Step 2 (FIXED) — Feature preparation
# Replaces station_code/train_no StringIndexer with frequency encoding
# to stay within maxBins=256 budget and avoid the 7616-values crash
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.functions import (
    col, when, isnan, isnull, coalesce, count, lit
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

LABEL = "arrival_delay_min"

# ── 1. Drop rows with null label ──────────────────────────────
df_clean = df.filter(col(LABEL).isNotNull())

# ── 2. Frequency-encode high-cardinality columns ──────────────
# Instead of StringIndexer (requires maxBins ≥ cardinality),
# we encode train_no and station_code as their frequency rank.
# This is a standard, compact encoding that preserves information.

def freq_encode(df, col_name: str, new_col: str):
    """Replace a string column with its frequency (count) across the dataset."""
    counts = (df.groupBy(col_name)
                .agg(count("*").alias("__cnt"))
                .withColumnRenamed(col_name, f"__{col_name}"))
    return (df
            .join(counts, df[col_name] == counts[f"__{col_name}"], "left")
            .withColumn(new_col, coalesce(col("__cnt"), lit(1)).cast("double"))
            .drop("__cnt", f"__{col_name}"))

df_enc = freq_encode(df_clean, "train_no", "train_no_freq")
df_enc = freq_encode(df_enc,   "station_code", "station_code_freq")

# ── 3. Fill nulls in numeric features ─────────────────────────
fill_dict = {
    "stop_seq":              0,
    "total_stops":           50,
    "cumulative_travel_min": 0,
    "dwell_min":             5,
    "scheduled_hour":        12,
    "is_peak_hour":          0,
    "pm25":                  60.0,
    "no2":                   25.0,
    "journey_day":           1,
    "train_no_freq":         1.0,
    "station_code_freq":     1.0,
}

# Only fill columns that exist
fill_dict_avail = {k: v for k, v in fill_dict.items() if k in df_enc.columns}
df_filled = df_enc.fillna(fill_dict_avail)

# ── 4. Low-cardinality categoricals only (safe for maxBins=256) ─
# train_type  → ~20 unique values  ✅
# zone        → ~17 unique values  ✅
LOW_CARD_CATS = []
for c in ["train_type", "zone"]:
    if c in df_filled.columns:
        n = df_filled.select(c).distinct().count()
        print(f"  {c}: {n} unique values")
        if n <= 250:
            LOW_CARD_CATS.append(c)
        else:
            print(f"  ⚠️  Skipping {c} — too many values ({n}) for maxBins=256")

# ── 5. Final numeric feature list ─────────────────────────────
NUMERIC_FEATURES = [
    "train_no_freq",        # frequency of this train (proxy for route importance)
    "station_code_freq",    # frequency of this station (proxy for junction size)
    "stop_seq",
    "total_stops",
    "cumulative_travel_min",
    "dwell_min",
    "scheduled_hour",
    "is_peak_hour",
    "pm25",
    "no2",
    "journey_day",
]
numeric_features_avail = [f for f in NUMERIC_FEATURES if f in df_filled.columns]

print(f"\n✅ Numeric features ({len(numeric_features_avail)}): {numeric_features_avail}")
print(f"✅ Low-cardinality categoricals: {LOW_CARD_CATS}")
print(f"✅ Clean rows: {df_filled.count():,}")

In [0]:
# ═══════════════════════════════════════════════════════════════
# Step 3 — Build Pipeline (MEMORY-SAFE + TUNED)
# Reduced maxIter + maxDepth to fit within 1GB Spark Connect cache
# ═══════════════════════════════════════════════════════════════

import os, gc, mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import lit, abs as abs_, col

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/setu_rail/bronze/raw_files"

LABEL = "arrival_delay_min"

# StringIndexers for low-cardinality only
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in LOW_CARD_CATS
]
indexed_cols     = [f"{c}_idx" for c in LOW_CARD_CATS]
all_feature_cols = numeric_features_avail + indexed_cols

assembler = VectorAssembler(
    inputCols     = all_feature_cols,
    outputCol     = "features",
    handleInvalid = "keep",
)

# ── Memory-safe GBT config ────────────────────────────────────
# maxIter=80, maxDepth=5 keeps model well under 1GB in Spark Connect
# Still better than original (was maxIter=100, maxDepth=6)
# minInstancesPerNode=20 prunes leaves aggressively → better MAE
gbt = GBTRegressor(
    featuresCol           = "features",
    labelCol              = LABEL,
    maxIter               = 80,       # safe for 1GB cache limit
    maxDepth              = 5,        # shallower = smaller model in memory
    stepSize              = 0.05,     # lower LR compensates for fewer trees
    maxBins               = 256,
    subsamplingRate       = 0.7,
    featureSubsetStrategy = "sqrt",
    minInstancesPerNode   = 20,       # aggressive pruning = better generalization
    seed                  = 42,
)

pipeline = Pipeline(stages=indexers + [assembler, gbt])

# ── Write to Delta first — avoids lazy recomputation blowing memory ──
print("💾 Persisting train/test splits to Delta...")
df_filled.randomSplit([0.8, 0.2], seed=42)[0] \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("setu_rail.gold._train_tmp")

df_filled.randomSplit([0.8, 0.2], seed=42)[1] \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("setu_rail.gold._test_tmp")

# Load back from Delta — clean, no lineage chains
train_df = spark.table("setu_rail.gold._train_tmp")
test_df  = spark.table("setu_rail.gold._test_tmp")

print(f"✅ Train: {train_df.count():,}  |  Test: {test_df.count():,}")
print(f"✅ Features ({len(all_feature_cols)}): {all_feature_cols}")

## Step 4 — Train and log to MLflow

In [0]:
# ═══════════════════════════════════════════════════════════════
# Step 4 — Train + Log (single fit, no retry loops)
# ═══════════════════════════════════════════════════════════════

with mlflow.start_run(run_name="gbt_v5_memory_safe") as run:
    print("🚀 Training GBT model...")
    model = pipeline.fit(train_df)
    print("✅ Training complete")

    preds = model.transform(test_df)

    mae  = RegressionEvaluator(labelCol=LABEL, metricName="mae").evaluate(preds)
    rmse = RegressionEvaluator(labelCol=LABEL, metricName="rmse").evaluate(preds)
    r2   = RegressionEvaluator(labelCol=LABEL, metricName="r2").evaluate(preds)

    print(f"\n📈 Metrics:")
    print(f"   MAE:  {mae:.2f} min")
    print(f"   RMSE: {rmse:.2f} min")
    print(f"   R²:   {r2:.3f}")

    mlflow.log_params({
        "maxIter": 80, "maxDepth": 5, "stepSize": 0.05,
        "maxBins": 256, "subsamplingRate": 0.7,
        "minInstancesPerNode": 20, "encoding": "frequency",
        "num_features": len(all_feature_cols),
    })
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

    # Signature from schema — proven to work
    input_schema = Schema([
        ColSpec("double", c) if str(train_df.schema[c].dataType) in
            ("DoubleType()", "FloatType()", "LongType()", "IntegerType()")
        else ColSpec("string", c)
        for c in (numeric_features_avail + LOW_CARD_CATS)
        if c in [f.name for f in train_df.schema.fields]
    ])
    signature = ModelSignature(
        inputs  = input_schema,
        outputs = Schema([ColSpec("double", "prediction")])
    )

    mlflow.spark.log_model(
        spark_model           = model,
        artifact_path         = "setu_gbt_model",
        registered_model_name = "setu_rail.gold.setu_delay_predictor",
        dfs_tmpdir            = "/Volumes/setu_rail/bronze/raw_files",
        signature             = signature,
    )
    print(f"✅ Run ID: {run.info.run_id}")
    print(f"✅ Registered: setu_rail.gold.setu_delay_predictor")

In [0]:
# ═══════════════════════════════════════════════════════════════
# Step 5 — Baseline + alias + cleanup
# ═══════════════════════════════════════════════════════════════

from mlflow import MlflowClient

# Baseline comparison
mean_delay   = train_df.agg({LABEL: "avg"}).collect()[0][0]
base_preds   = test_df.withColumn("prediction", lit(mean_delay))
baseline_mae = base_preds.withColumn("error", abs_(col(LABEL) - col("prediction"))) \
                         .agg({"error": "avg"}).collect()[0][0]
improvement  = 100 * (baseline_mae - mae) / baseline_mae

print(f"Baseline MAE: {baseline_mae:.2f} min")
print(f"GBT MAE:      {mae:.2f} min")
print(f"🎯 Improvement: {improvement:.1f}%")

# Set @production alias
client   = MlflowClient()
versions = client.search_model_versions("name='setu_rail.gold.setu_delay_predictor'")
latest   = max(versions, key=lambda v: int(v.version))
client.set_registered_model_alias(
    name    = "setu_rail.gold.setu_delay_predictor",
    alias   = "production",
    version = latest.version,
)
print(f"✅ @production alias → version {latest.version}")

# Cleanup
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._train_tmp")
spark.sql("DROP TABLE IF EXISTS setu_rail.gold._test_tmp")
print("✅ Done — ready for 03_shap_explainability")

## Step 5 — Baseline comparison

In [0]:
# What if we just always predicted the mean?
from pyspark.sql.functions import lit, abs as abs_

mean_delay = train_df.agg({LABEL: "avg"}).collect()[0][0]
print(f"Mean delay in training set: {mean_delay:.2f} min")

# Compute baseline MAE
base_preds = test_df.withColumn("prediction", lit(mean_delay))
baseline_mae = base_preds.withColumn("error", abs_(col(LABEL) - col("prediction"))) \
                         .agg({"error": "avg"}).collect()[0][0]

improvement = 100 * (baseline_mae - mae) / baseline_mae
print(f"\nBaseline (always-predict-mean) MAE: {baseline_mae:.2f} min")
print(f"GBT model MAE:                       {mae:.2f} min")
print(f"🎯 Improvement:                      {improvement:.1f}%")

## Step 6 — Save predictions for dashboard

In [0]:
# Select key columns for the dashboard
cols_to_keep = ["train_no", "station_code", "scheduled_hour",
                 col(LABEL).alias("actual_delay_min"),
                 col("prediction").alias("predicted_delay_min")]

# Add optional columns if they exist
if "pm25" in preds.columns:
    cols_to_keep.append("pm25")
if "no2" in preds.columns:
    cols_to_keep.append("no2")
if "zone" in preds.columns:
    cols_to_keep.append("zone")

predictions_daily = preds.select(cols_to_keep)

(predictions_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.gold.predictions_daily"))

print(f"✅ Saved {predictions_daily.count():,} predictions to gold.predictions_daily")

## Step 7 — Create production alias in MLflow

## Summary

✅ **Model trained & logged to UC**
- Metrics: MAE {mae:.2f}min, RMSE {rmse:.2f}min
- Used: {len(categorical_features_avail)} categorical + {len(numeric_features_avail)} numeric features
- Distributed via Spark MLlib + Delta partitioning

✅ **Next:** `03_shap_explainability.ipynb`